In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.5.1+cu121
12.1
True
NVIDIA TITAN Xp


In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Count: 1
GPU Name: NVIDIA TITAN Xp


In [3]:
import bitsandbytes as bnb

print(bnb.__version__)

0.49.2


In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

model_name = "Qwen/Qwen3-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loaded")

/mnt/sda1/osint_intern/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 398/398 [02:10<00:00,  3.04it/s]


Loaded


In [5]:
print(next(model.parameters()).device)

cpu


In [6]:
import torch

print(torch.cuda.memory_allocated()/1024**3, "GB")
print(torch.cuda.memory_reserved()/1024**3, "GB")

0.0 GB
0.001953125 GB


In [7]:
import pandas as pd



In [8]:
df=pd.read_csv("/mnt/sda1/osint_intern/SA_Group/Raunak/data.csv")
df
df.columns

Index(['Sentence', 'Region', 'State'], dtype='str')

In [9]:
df.head(10)


,Sentence,Region,State
0,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Northern Cultural Region,Himachal Pradesh
1,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Northern Cultural Region,Haryana
2,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Northern Cultural Region,Uttarakhand
3,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Southern Cultural Region,Andhra Pradesh
4,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Southern Cultural Region,Karnataka
5,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Southern Cultural Region,Lakshadweep
6,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Western Cultural Region,Maharashtra
7,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Western Cultural Region,Daman and Diu
8,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Western Cultural Region,Dadra and Nagar Haveli
9,कई नवविवाहित जोड़े शादी के बाद अपने विस्तारित ...,Eastern Cultural Region,Odisha


In [10]:
print("Done")

Done


In [11]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [12]:
!nvidia-smi

Wed Jun 24 12:55:51 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA TITAN Xp                Off | 00000000:01:00.0 Off |                  N/A |
| 54%   83C    P2             174W / 250W |  10848MiB / 12288MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [12]:
import pandas as pd
import json
import re
import torch
from tqdm.auto import tqdm

# =====================================================
# PROMPT
# =====================================================

PROMPT_TEMPLATE = """
You are an expert sentiment classifier.

Your task is to classify sentiment by considering:
- cultural context
- regional practices
- social norms
- geographical sensitivity
- local interpretation

A sentence may express different sentiment in different locations.

STRICT RULES:
- Output ONLY valid JSON
- No explanation
- No markdown
- No extra text

SCHEMA:
{{
    "sentiment":"positive | neutral | negative"
}}

Location:
{location}

Text:
{text}

Return ONLY JSON.
"""

# =====================================================
# SETTINGS
# =====================================================

BATCH_SIZE = 8

# =====================================================
# JSON PARSER
# =====================================================

def extract_sentiment(response):

    response = response.strip()

    try:

        match = re.search(
            r"\{.*?\}",
            response,
            re.DOTALL
        )

        if match:

            obj = json.loads(
                match.group()
            )

            sentiment = (
                obj["sentiment"]
                .strip()
                .lower()
            )

            if sentiment in [
                "positive",
                "neutral",
                "negative"
            ]:
                return sentiment

    except:
        pass

    text = response.lower()

    if "positive" in text:
        return "positive"

    if "neutral" in text:
        return "neutral"

    if "negative" in text:
        return "negative"

    return "PARSE_ERROR"


In [13]:
# =====================================================
# QWEN INFERENCE
# =====================================================

def classify_batch(batch_df):

    prompts = []

    for _, row in batch_df.iterrows():

        location = (
            f"{row['State']} - "
            f"{row['Region']}"
        )

        prompt = PROMPT_TEMPLATE.format(
            location=location,
            text=str(row["Sentence"])
        )

        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    sentiments = []

    for i in range(len(prompts)):

        generated_tokens = outputs[i][
            inputs["input_ids"][i].shape[0]:
        ]

        decoded = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

        sentiments.append(
            extract_sentiment(decoded)
        )

    return sentiments


In [ ]:
# =====================================================
# MAIN PROCESSING
# =====================================================

all_predictions = []

for start in tqdm(
    range(0, len(df), BATCH_SIZE),
    desc="Classifying"
):

    batch_df = df.iloc[
        start:start+BATCH_SIZE
    ]

    preds = classify_batch(
        batch_df
    )

    all_predictions.extend(preds)


Classifying:   0%|                                                                                                                                                                       | 0/1479 [00:00<?, ?it/s]

In [ ]:
import os

NOTEBOOK_DIR = os.getcwd()

for region in output_df["Region"].unique():

    region_df = output_df[
        output_df["Region"] == region
    ].copy()

    filename = os.path.join(
        NOTEBOOK_DIR,
        region.replace(" ", "_") + "Qwen3._4B.csv"
    )

    region_df.to_csv(
        filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved -> {filename}")

print("\nFinished.")